In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv('jiangxi_causal_forest_panel_1km.csv')
base_cols = ['ntl', 'pop', 'GDP_log', 'people_hea', 'road', 'elder', 'sports_cnt']
year_list = sorted(df['year'].unique())

feat_imp_years = []
oob_scores = []

for year in year_list:
    df_year = df[df['year'] == year].copy()
    X = df_year[base_cols]
    y = df_year['medical_cnt'].values.ravel()
    rf = RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=10,
        n_jobs=-1,
        random_state=0,
        oob_score=True
    )
    rf.fit(X, y)
    # 特征重要性与OOB
    feat_imp_years.append(rf.feature_importances_)
    oob_scores.append(rf.oob_score_)

feat_imp_df = pd.DataFrame(feat_imp_years, columns=base_cols, index=year_list)
feat_imp_df.index.name = 'year'
feat_imp_df['oob_score'] = oob_scores
feat_imp_df = feat_imp_df.reset_index()
feat_imp_df.to_csv('feature_importance_by_year_rf.csv', index=False, encoding='utf-8-sig')
print(feat_imp_df)


    year       ntl       pop   GDP_log  people_hea      road     elder  \
0   2012  0.149588  0.186768  0.015106    0.019627  0.048027  0.025876   
1   2013  0.152474  0.167724  0.011071    0.018340  0.017253  0.046448   
2   2014  0.149091  0.178147  0.013133    0.016694  0.019513  0.044498   
3   2015  0.132064  0.172983  0.010681    0.013730  0.018044  0.042529   
4   2016  0.121094  0.164154  0.012557    0.028436  0.018672  0.026428   
5   2017  0.112238  0.159177  0.008807    0.011879  0.025354  0.038547   
6   2018  0.102114  0.161574  0.010377    0.010885  0.015050  0.037165   
7   2019  0.098863  0.165492  0.009882    0.012789  0.018791  0.032431   
8   2020  0.087795  0.166591  0.014084    0.024990  0.013971  0.023896   
9   2021  0.081638  0.160365  0.012548    0.024261  0.021764  0.018748   
10  2022  0.072140  0.158047  0.010234    0.028332  0.024077  0.018181   
11  2023  0.085145  0.120392  0.013088    0.022342  0.021332  0.016933   

    sports_cnt  oob_score  
0     0.5

In [2]:
import pandas as pd
import xgboost as xgb

df = pd.read_csv('jiangxi_causal_forest_panel_1km.csv')
base_cols = ['ntl', 'pop', 'GDP_log', 'people_hea', 'road', 'elder', 'sports_cnt']
year_list = sorted(df['year'].unique())

feat_imp_years = []
eval_scores = []

for year in year_list:
    df_year = df[df['year'] == year].copy()
    X = df_year[base_cols]
    y = df_year['medical_cnt'].values.ravel()
    dtrain = xgb.DMatrix(X, label=y, feature_names=base_cols)
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'seed': 0
    }
    bst = xgb.train(params, dtrain, num_boost_round=100, verbose_eval=False)
    # importance_type 可选 'weight', 'gain', 'cover' 等
    imp = bst.get_score(importance_type='gain')
    # 归一化为和为1
    imp_vec = [imp.get(f, 0.0) for f in base_cols]
    imp_sum = sum(imp_vec)
    imp_vec = [v/imp_sum if imp_sum>0 else 0 for v in imp_vec]
    feat_imp_years.append(imp_vec)
    # 用训练集自身做一次分数（仅做参考，不如oob有外推意义）
    pred = bst.predict(dtrain)
    r2 = 1 - ((y - pred)**2).sum() / ((y - y.mean())**2).sum()
    eval_scores.append(r2)

feat_imp_df = pd.DataFrame(feat_imp_years, columns=base_cols, index=year_list)
feat_imp_df.index.name = 'year'
feat_imp_df['train_r2'] = eval_scores
feat_imp_df = feat_imp_df.reset_index()
feat_imp_df.to_csv('feature_importance_by_year_xgb.csv', index=False, encoding='utf-8-sig')
print(feat_imp_df)


    year       ntl       pop   GDP_log  people_hea      road     elder  \
0   2012  0.080497  0.094076  0.077711    0.084074  0.116200  0.090781   
1   2013  0.080135  0.085606  0.083276    0.079858  0.065730  0.123789   
2   2014  0.073505  0.085389  0.065408    0.086477  0.075736  0.105937   
3   2015  0.073738  0.086014  0.074435    0.074996  0.064322  0.112459   
4   2016  0.069247  0.085723  0.076784    0.096626  0.054688  0.093695   
5   2017  0.068977  0.087354  0.063608    0.056425  0.093572  0.121985   
6   2018  0.061030  0.094563  0.073994    0.063520  0.082285  0.102807   
7   2019  0.066063  0.085413  0.066797    0.055660  0.107906  0.100591   
8   2020  0.058281  0.089007  0.073534    0.090094  0.069452  0.068116   
9   2021  0.066071  0.088909  0.084489    0.096315  0.067991  0.063885   
10  2022  0.056868  0.084727  0.060991    0.097458  0.077245  0.076405   
11  2023  0.071092  0.077949  0.076483    0.078980  0.070835  0.080541   

    sports_cnt  train_r2  
0     0.45

In [4]:
import pandas as pd
import xgboost as xgb
import shap
import numpy as np
df = pd.read_csv('jiangxi_causal_forest_panel_1km.csv')
base_cols = ['ntl', 'pop', 'GDP_log', 'people_hea', 'road', 'elder', 'sports_cnt']
year_list = sorted(df['year'].unique())

shap_imp_years = []
mean_abs_shap_years = []

for year in year_list:
    df_year = df[df['year'] == year].copy()
    X = df_year[base_cols]
    y = df_year['medical_cnt'].values.ravel()
    dtrain = xgb.DMatrix(X, label=y, feature_names=base_cols)
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'seed': 0
    }
    bst = xgb.train(params, dtrain, num_boost_round=100, verbose_eval=False)
    # SHAP分析
    explainer = shap.Explainer(bst, X, feature_names=base_cols)
    shap_values = explainer(X)
    # 各特征mean(|SHAP|)作为全局重要性
    mean_abs_shap = pd.DataFrame({
        'feature': base_cols,
        'mean_abs_shap': np.abs(shap_values.values).mean(axis=0)
    })
    mean_abs_shap['year'] = year
    mean_abs_shap_years.append(mean_abs_shap)

# 合并所有年份结果
all_shap_df = pd.concat(mean_abs_shap_years, axis=0)
# 以表格形式保存
all_shap_df_pivot = all_shap_df.pivot(index='year', columns='feature', values='mean_abs_shap').reset_index()
all_shap_df_pivot.to_csv('shap_feature_importance_by_year.csv', index=False, encoding='utf-8-sig')
print(all_shap_df_pivot)


 99%|===================| 23650/23795 [00:54<00:00]        

feature  year   GDP_log     elder       ntl  people_hea       pop      road  \
0        2012  0.184044  0.116416  0.625310    0.074496  0.700223  0.247283   
1        2013  0.105511  0.218490  0.609198    0.078519  0.527166  0.072622   
2        2014  0.132009  0.214771  0.750538    0.160090  0.665950  0.082642   
3        2015  0.214020  0.258678  0.708300    0.060095  0.720735  0.095213   
4        2016  0.203209  0.221246  0.816284    0.233139  0.743316  0.093884   
5        2017  0.159256  0.194083  0.627947    0.042934  0.632549  0.146536   
6        2018  0.152415  0.222352  0.633013    0.040719  0.636359  0.072560   
7        2019  0.126326  0.152867  0.729014    0.061533  0.597842  0.130450   
8        2020  0.169570  0.110798  0.601153    0.158702  0.891641  0.055206   
9        2021  0.165496  0.088215  0.498322    0.152202  0.569130  0.087018   
10       2022  0.102328  0.072936  0.366601    0.134102  0.670786  0.074236   
11       2023  0.127910  0.051220  0.494652    0.130

In [7]:
import pandas as pd
import xgboost as xgb
import shap
import os
from shapely.geometry import Point
import geopandas as gpd
from tqdm import tqdm

df = pd.read_csv('jiangxi_causal_forest_panel_1km.csv')
base_cols = ['ntl', 'pop', 'GDP_log', 'people_hea', 'road', 'elder', 'sports_cnt']
out_folder = "yearly_shap_maps"
os.makedirs(out_folder, exist_ok=True)

for year in sorted(df['year'].unique()):
    print(f"处理年份：{year}")
    df_year = df[df['year']==year].copy()
    X = df_year[base_cols]
    y = df_year['medical_cnt'].values.ravel()
    dtrain = xgb.DMatrix(X, label=y, feature_names=base_cols)
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'seed': 0
    }
    bst = xgb.train(params, dtrain, num_boost_round=100, verbose_eval=False)
    explainer = shap.Explainer(bst, X, feature_names=base_cols)
    shap_values = explainer(X)
    # 取sports_cnt变量的SHAP值
    sports_shap = shap_values.values[:, base_cols.index('sports_cnt')]
    df_year['sports_cnt_shap'] = sports_shap

    # 修正geometry: 保证为Point对象
    if ('geometry' in df_year.columns and isinstance(df_year['geometry'].iloc[0], Point)):
        gdf = gpd.GeoDataFrame(df_year, geometry='geometry')
    elif {'x', 'y'}.issubset(df_year.columns):
        gdf = gpd.GeoDataFrame(df_year, geometry=[Point(xy) for xy in zip(df_year['x'], df_year['y'])])
    elif {'lng', 'lat'}.issubset(df_year.columns):
        gdf = gpd.GeoDataFrame(df_year, geometry=[Point(xy) for xy in zip(df_year['lng'], df_year['lat'])])
    elif {'col', 'row'}.issubset(df_year.columns):
        # 注意col/row如果是行列号而不是经纬度，不能用于地图，需要真实坐标！
        gdf = gpd.GeoDataFrame(df_year, geometry=[Point(xy) for xy in zip(df_year['col'], df_year['row'])])
    else:
        raise ValueError("没有有效坐标列！请检查你的数据包含 x/y 或 lng/lat 或 geometry 列。")
    # 设置坐标系（如有真实经纬度，建议为4326）
    gdf.set_crs('EPSG:4326', inplace=True, allow_override=True)

    # 输出GeoPackage，也可以保存为shp或csv
    gdf.to_file(f"{out_folder}/{year}_sports_cnt_shap.gpkg", driver='GPKG', encoding='utf-8')

    print(f"已保存 {out_folder}/{year}_sports_cnt_shap.gpkg")

print("全部年份SHAP空间分布已保存完毕！")


处理年份：2012


 97%|=================== | 4744/4872 [00:16<00:00]       

已保存 yearly_shap_maps/2012_sports_cnt_shap.gpkg
处理年份：2013


 97%|=================== | 7635/7836 [00:24<00:00]       

已保存 yearly_shap_maps/2013_sports_cnt_shap.gpkg
处理年份：2014


...

已保存 yearly_shap_maps/2021_sports_cnt_shap.gpkg
处理年份：2022


100%|===================| 19853/19936 [00:49<00:00]        

已保存 yearly_shap_maps/2022_sports_cnt_shap.gpkg
处理年份：2023


 99%|===================| 23618/23795 [00:58<00:00]        

已保存 yearly_shap_maps/2023_sports_cnt_shap.gpkg
全部年份SHAP空间分布已保存完毕！


In [8]:
import pandas as pd
import xgboost as xgb
import shap
import numpy as np
import matplotlib.pyplot as plt
import os

df = pd.read_csv('jiangxi_causal_forest_panel_1km.csv')
base_cols = ['ntl', 'pop', 'GDP_log', 'people_hea', 'road', 'elder', 'sports_cnt']
year_list = sorted(df['year'].unique())

outdir = 'shap_summary_plots'
os.makedirs(outdir, exist_ok=True)

for year in year_list:
    print(f"画图年份：{year}")
    df_year = df[df['year'] == year].copy()
    X = df_year[base_cols]
    y = df_year['medical_cnt'].values.ravel()
    dtrain = xgb.DMatrix(X, label=y, feature_names=base_cols)
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'seed': 0
    }
    bst = xgb.train(params, dtrain, num_boost_round=100, verbose_eval=False)
    # SHAP分析
    explainer = shap.Explainer(bst, X, feature_names=base_cols)
    shap_values = explainer(X)
    
    # summary plot保存为PNG
    plt.figure()
    shap.summary_plot(
        shap_values.values, X, feature_names=base_cols,
        show=False,  # 不自动展示，直接保存
        plot_size=(8,6)
    )
    plt.title(f"{year}年 SHAP summary plot", fontname='SimSun', fontsize=16)
    plt.tight_layout()
    plt.savefig(f"{outdir}/shap_summary_{year}.png", dpi=300, bbox_inches='tight')
    plt.close()

print(f"全部summary plot已保存到：{outdir}/")


画图年份：2012


 95%|=================== | 4629/4872 [00:14<00:00]       

画图年份：2013


 97%|=================== | 7617/7836 [00:21<00:00]       

画图年份：2014


...

画图年份：2022


 99%|===================| 19669/19936 [00:50<00:00]        

画图年份：2023


 99%|===================| 23469/23795 [00:55<00:00]        

全部summary plot已保存到：shap_summary_plots/
